# Chaîne ETL du datamart BTK

Consolidation des données opérationnelles en un **datamart en étoile agrégé par
agence**, qui alimente le tableau de bord, Power BI et la segmentation.

> **Exécution : menu *Kernel → Restart Kernel and Run All Cells*.**
> Les cellules se lisent de haut en bas ; la première (« 0. Préparation »)
> définit les imports et le chemin du projet, elle doit donc passer en premier.

Ce notebook est autonome : il n'importe aucun module du projet. Il doit
seulement être placé **dans le dossier du projet**, à côté des dossiers `etl` et
`clustering`, pour y trouver les données.

La source est choisie automatiquement, dans cet ordre :
1. **Oracle** — si les variables `BTK_DB_USER`, `BTK_DB_PWD`, `BTK_DB_DSN` sont définies ;
2. **CSV** — si les cinq exports de `etl/export_sources.sql` sont dans `etl/source/` ;
3. **extrait agrégé réel** — le relevé des 49 entités du réseau livré avec le projet.

## 0. Préparation

In [1]:
import os, sys
import numpy as np
import pandas as pd

# On remonte depuis le dossier courant jusqu'à celui qui contient à la fois
# les répertoires "etl" et "clustering" : c'est la racine du projet.
_dossier = os.path.abspath(os.getcwd())
while not all(os.path.isdir(os.path.join(_dossier, d)) for d in ("etl", "clustering")):
    _parent = os.path.dirname(_dossier)
    if _parent == _dossier:
        raise FileNotFoundError(
            "Racine du projet introuvable depuis " + os.getcwd() + "\n"
            "Placez ce notebook dans le dossier du projet, à côté des dossiers "
            "« etl » et « clustering », puis relancez cette cellule.")
    _dossier = _parent
RACINE = _dossier

pd.set_option("display.width", 170, "display.max_columns", 20)
print("Racine du projet :", RACINE)
print("pandas", pd.__version__, "| numpy", np.__version__)

Racine du projet : /home/user/oussema_korbosli
pandas 3.0.5 | numpy 2.4.6


## 1. Connexion à la base Oracle

Prérequis, une seule fois, dans l'**invite Anaconda** : `pip install oracledb`.
Le pilote fonctionne en mode *thin* : aucun client Oracle à installer.

Les paramètres par défaut sont ceux de l'application
(`src/main/resources/META-INF/persistence.xml`). Le mot de passe n'est pas
écrit dans le notebook : il est demandé à la saisie.

Si la connexion échoue, la cellule affiche la cause probable et **le notebook
continue** sur l'extrait agrégé réel livré avec le projet.

In [2]:
if "RACINE" not in globals():
    raise RuntimeError("Exécutez d'abord la cellule « 0. Préparation ». "
                       "Le plus simple : menu Kernel > Restart Kernel and Run All Cells.")

UTILISER_ORACLE = True   # False -> travailler sur l'extrait livré, sans invite

BTK_USER = os.environ.get("BTK_DB_USER", "SYSTEM")
BTK_DSN  = os.environ.get("BTK_DB_DSN",  "localhost:1521/FREEPDB1")
BTK_PWD  = os.environ.get("BTK_DB_PWD")

DIAGNOSTIC = {
    "ORA-01017": "identifiant ou mot de passe incorrect.",
    "ORA-12541": "aucun listener : le service Oracle n'est pas démarré "
                 "(Windows : services.msc, démarrer OracleServiceFREE et le TNSListener).",
    "DPY-6005":  "connexion refusée : vérifiez l'hôte et le port du DSN, et que "
                 "la base est bien démarrée.",
    "ORA-12514": "nom de service inconnu : essayez XEPDB1 ou ORCLPDB1 à la place "
                 "de FREEPDB1 (le nom dépend de la version d'Oracle installée).",
    "ORA-12154": "nom de service introuvable : donnez le DSN complet hote:port/service.",
    "ORA-28000": "compte verrouillé : ALTER USER ... ACCOUNT UNLOCK.",
}

CONNEXION = None
if not UTILISER_ORACLE:
    print("Oracle désactivé (UTILISER_ORACLE = False).")
else:
  try:
    import oracledb
  except ImportError:
    print("Le pilote « oracledb » n'est pas installé.")
    print("Dans l'invite Anaconda :  pip install oracledb")
  else:
    if not BTK_PWD:
        from getpass import getpass
        BTK_PWD = getpass(f"Mot de passe Oracle de {BTK_USER}@{BTK_DSN} : ")
    try:
        CONNEXION = oracledb.connect(user=BTK_USER, password=BTK_PWD, dsn=BTK_DSN)
        print(f"Connecté à {BTK_USER}@{BTK_DSN}")
        print("Serveur Oracle", CONNEXION.version)
    except Exception as err:
        print("Connexion impossible :", str(err).splitlines()[0])
        for cle, conseil in DIAGNOSTIC.items():
            if cle in str(err):
                print("  ->", conseil)
                break

if CONNEXION is None:
    print("\nLa suite du notebook utilisera l'extrait agrégé réel du réseau.")

Connexion impossible : DPY-6005: cannot connect to database (CONNECTION_ID=UNR9SweqwiUBtRddi9Gk+Q==).
  -> connexion refusée : vérifiez l'hôte et le port du DSN, et que la base est bien démarrée.

La suite du notebook utilisera l'extrait agrégé réel du réseau.


### Inventaire des tables sources

Contrôle de lecture table par table : une table illisible n'interrompt pas
l'inventaire, ce qui permet de voir d'un coup d'œil ce qui manque.

In [3]:
# Exécute une requête et renvoie un DataFrame (colonnes en majuscules).
def q(sql):
    with CONNEXION.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

if CONNEXION is None:
    print("Pas de connexion : inventaire ignoré.")
    inventaire = None
else:
    lignes = []
    for t in ["AGENCE", "B_UTILISATEURS", "CLIENT_BTK", "B_OBJECTIF", "POINTAGE"]:
        try:
            lignes.append([t, int(q(f"SELECT COUNT(*) AS N FROM {t}")["N"][0]), "OK"])
        except Exception as err:
            lignes.append([t, None, str(err).splitlines()[0][:45]])
    inventaire = pd.DataFrame(lignes, columns=["Table source", "Lignes", "État"])
inventaire

Pas de connexion : inventaire ignoré.


## 2. Collecte (*Extract*)

Lecture des cinq tables sources. `POINTAGE` est traitée à part : si elle n'a pas
encore été créée (`sql/setup_pointage.sql`), la chaîne continue sans le taux de
présence.

Sans connexion Oracle, la collecte se rabat sur l'export CSV
(`etl/source/`) puis, à défaut, sur l'extrait agrégé réel.

In [4]:
if "CONNEXION" not in globals():
    raise RuntimeError("Exécutez d'abord les cellules 0 et 1 "
                       "(Préparation et Connexion).")

SOURCE  = os.path.join(RACINE, "etl", "source")
EXTRAIT = os.path.join(RACINE, "clustering", "data", "agences_reelles.csv")
TABLES  = ["agences", "employes", "clients", "objectifs", "pointages"]

SQL = {
    "agences":   "SELECT SK_AGENCE, LIBELLE_AGENCE, DISTRICT FROM AGENCE",
    "employes":  "SELECT SK_UTILISATEUR, LIBELLE_UTILISATEUR, SK_AGENCE, "
                 "EST_GESTIONNAIRE FROM B_UTILISATEURS",
    "clients":   "SELECT SK_CLIENT, SK_AGENCE FROM CLIENT_BTK",
    "objectifs": "SELECT SK_AGENCE, SK_UTILISATEUR, "
                 "  NVL(SOUSCRIPTION_COMPTE_CHEQUES_OA,0)"
                 "  + NVL(SOUSCRIPTION_COMPTE_EPARGNES_OA,0)"
                 "  + NVL(SOUSCRIPTION_COMPTE_COURANTS_OA,0)      AS COMPTES, "
                 "  NVL(PRODUCTION_CREDITS_CONSO_OA,0)"
                 "  + NVL(PRODUCTION_CREDITS_IMMO_OA,0)"
                 "  + NVL(PRODUCTION_CREDITS_INVESTISSEMENT_OA,0) AS CREDITS, "
                 "  NVL(EPARGNE_ADD_OA,0)                         AS EPARGNE "
                 "FROM B_OBJECTIF",
    "pointages": "SELECT P.SK_UTILISATEUR, U.SK_AGENCE, P.STATUT FROM POINTAGE P "
                 "JOIN B_UTILISATEURS U ON P.SK_UTILISATEUR = U.SK_UTILISATEUR",
}

if CONNEXION is not None:                               # 1) base Oracle
    tables, mode = {}, "brut"
    for n in TABLES:
        try:
            tables[n] = q(SQL[n])
        except Exception as err:
            if n != "pointages":
                raise
            print("POINTAGE illisible :", str(err).splitlines()[0][:60])
            print("  -> le taux de présence ne sera pas calculé.")
    print("source : base Oracle BTK")
elif all(os.path.exists(os.path.join(SOURCE, n + ".csv")) for n in TABLES):   # 2) CSV
    tables = {n: pd.read_csv(os.path.join(SOURCE, n + ".csv")) for n in TABLES}
    mode = "brut"
    print("source : export CSV des tables —", os.path.relpath(SOURCE, RACINE))
else:                                                   # 3) extrait agrégé réel
    extrait = pd.read_csv(EXTRAIT)
    extrait.insert(0, "SK_AGENCE", range(1, len(extrait) + 1))
    tables, mode = {"extrait": extrait}, "extrait"
    print("source : extrait agrégé réel —", os.path.relpath(EXTRAIT, RACINE))

if mode == "brut":
    print(" | ".join(f"{n} : {len(tables[n])}" for n in TABLES if n in tables))
    apercu = tables["agences"].head(10)
else:
    print(len(tables["extrait"]), "entités du réseau")
    apercu = tables["extrait"].head(10)
apercu

source : extrait agrégé réel — clustering/data/agences_reelles.csv
49 entités du réseau


,SK_AGENCE,agence,nb_gestionnaires,effectif,nb_clients,total_comptes,production_credits,collecte_epargne
0,1,BIZERTE,8,10,1795,799,6300000,812500
1,2,MGHIRA,6,9,1791,607,5880000,690000
2,3,MONASTIR,10,12,1497,557,7000000,450000
3,4,GROMBALIA,6,12,1451,618,5400000,662500
4,5,BEN AROUS,7,18,1313,708,8300000,720000
5,6,CENTRALE,9,26,1226,528,59400000,420000
6,7,NABEUL,7,8,1225,669,6350000,840000
7,8,SFAX 2,6,15,1154,436,4750000,385000
8,9,LA MARSA,5,8,1126,618,6250000,662500
9,10,PALMARIUM,5,13,1118,712,8100000,732500


## 2. Nettoyage

Les enregistrements sans agence de rattachement (`SK_AGENCE` manquant) sont
écartés, les colonnes typées et les mesures converties en numérique.

*Étape sans objet lorsque la source est l'extrait déjà agrégé : il ne contient
qu'une ligne par agence, sans clé manquante.*

In [5]:
if mode == "brut":
    for n in [x for x in ["employes", "clients", "objectifs", "pointages"]
              if x in tables]:
        avant = len(tables[n])
        tables[n] = tables[n].dropna(subset=["SK_AGENCE"]).copy()
        tables[n]["SK_AGENCE"] = tables[n]["SK_AGENCE"].astype(int)
        print(f"{n:<10} {avant:>7} -> {len(tables[n]):>7} lignes")
    tables["employes"]["EST_GESTIONNAIRE"] = (
        tables["employes"]["EST_GESTIONNAIRE"].fillna(0).astype(int))
    for c in ["COMPTES", "CREDITS", "EPARGNE"]:
        tables["objectifs"][c] = pd.to_numeric(
            tables["objectifs"][c], errors="coerce").fillna(0)
else:
    print("Source déjà agrégée par agence : aucune ligne à écarter.")

Source déjà agrégée par agence : aucune ligne à écarter.


## 3 et 4. Transformation et intégration

Agrégation par agence, puis fusion des quatre jeux sur la clé `SK_AGENCE` :

| Mesure | Calcul |
|---|---|
| `effectif` | nombre d'employés de l'agence |
| `nb_gestionnaires` | employés avec `EST_GESTIONNAIRE = 1` |
| `nb_clients` | nombre de clients rattachés |
| `total_comptes` | somme des ouvertures de comptes |
| `production_credits` | somme de la production de crédits |
| `collecte_epargne` | somme de l'épargne additionnelle |
| `taux_presence` | part des pointages « présent » ou « retard » |

In [6]:
MESURES = ["effectif", "nb_gestionnaires", "nb_clients",
           "total_comptes", "production_credits", "collecte_epargne"]

if mode == "brut":
    eff = tables["employes"].groupby("SK_AGENCE").agg(
        effectif=("SK_UTILISATEUR", "count"),
        nb_gestionnaires=("EST_GESTIONNAIRE", "sum")).reset_index()
    cli = tables["clients"].groupby("SK_AGENCE").size().reset_index(name="nb_clients")
    obj = tables["objectifs"].groupby("SK_AGENCE").agg(
        total_comptes=("COMPTES", "sum"),
        production_credits=("CREDITS", "sum"),
        collecte_epargne=("EPARGNE", "sum")).reset_index()
    datamart = (tables["agences"].merge(eff, on="SK_AGENCE", how="left")
                                 .merge(cli, on="SK_AGENCE", how="left")
                                 .merge(obj, on="SK_AGENCE", how="left")
                                 .rename(columns={"LIBELLE_AGENCE": "agence"}))
    if "pointages" in tables:                    # taux de présence si POINTAGE existe
        poi = tables["pointages"]
        pres = poi.assign(present=poi["STATUT"].isin(["PRESENT", "RETARD"]).astype(int)) \
                  .groupby("SK_AGENCE").agg(taux_presence=("present", "mean")).reset_index()
        datamart = datamart.merge(pres, on="SK_AGENCE", how="left")
else:
    datamart = tables["extrait"].copy()          # déjà au grain de l'agence

for c in ["effectif", "nb_gestionnaires", "nb_clients"]:
    datamart[c] = pd.to_numeric(datamart[c], errors="coerce").fillna(0).astype(int)
for c in ["total_comptes", "production_credits", "collecte_epargne"]:
    datamart[c] = pd.to_numeric(datamart[c], errors="coerce").fillna(0).round(1)

mesures = list(MESURES)
if "taux_presence" in datamart.columns:
    datamart["taux_presence"] = pd.to_numeric(
        datamart["taux_presence"], errors="coerce").fillna(0).round(3)
    mesures.append("taux_presence")

datamart = datamart.sort_values("SK_AGENCE").reset_index(drop=True)
print(f"datamart : {len(datamart)} agences x {len(mesures)} mesures")
datamart.sort_values("nb_clients", ascending=False).head(10)[["agence"] + mesures]

datamart : 49 agences x 6 mesures


,agence,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
0,BIZERTE,10,8,1795,799,6300000,812500
1,MGHIRA,9,6,1791,607,5880000,690000
2,MONASTIR,12,10,1497,557,7000000,450000
3,GROMBALIA,12,6,1451,618,5400000,662500
4,BEN AROUS,18,7,1313,708,8300000,720000
5,CENTRALE,26,9,1226,528,59400000,420000
6,NABEUL,8,7,1225,669,6350000,840000
7,SFAX 2,15,6,1154,436,4750000,385000
8,LA MARSA,8,5,1126,618,6250000,662500
9,PALMARIUM,13,5,1118,712,8100000,732500


## 5. Contrôle de qualité

Avant chargement : aucune valeur manquante, aucune valeur négative, aucune
agence en double.

In [7]:
manquants = int(datamart[mesures].isna().sum().sum())
negatifs  = int((datamart[mesures] < 0).sum().sum())
doublons  = int(datamart["agence"].duplicated().sum())
print("valeurs manquantes :", manquants)
print("valeurs négatives  :", negatifs)
print("agences en double  :", doublons)
assert not (manquants or negatifs or doublons), "Anomalie : chargement interrompu."
datamart[mesures].describe().round(1)

valeurs manquantes : 0
valeurs négatives  : 0
agences en double  : 0


,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
count,49.0,49.0,49.0,49.0,49.0,49.0
mean,20.3,7.1,611.1,428.1,7718138.3,478939.3
std,76.0,10.4,566.3,252.4,10332885.3,517198.8
min,0.0,0.0,0.0,0.0,0.0,0.0
25%,5.0,4.0,2.0,333.0,3800000.0,210000.0
50%,10.0,6.0,724.0,502.0,5324000.0,457500.0
75%,14.0,8.0,1058.0,607.0,6500000.0,650000.0
max,540.0,76.0,1795.0,801.0,59400000.0,3508025.0


## 6. Chargement (*Load*)

Écriture du datamart en étoile dans `etl/entrepot/`, puis du fichier consommé
par la segmentation (`clustering/data/agences.csv`).

L'axe gestionnaire (`dim_gestionnaire`, `fait_objectif`) n'est produit que si la
source porte le détail par employé, c'est-à-dire Oracle ou l'export CSV.

In [8]:
ENTREPOT = os.path.join(RACINE, "etl", "entrepot")
DATAMART = os.path.join(RACINE, "clustering", "data", "agences.csv")
os.makedirs(ENTREPOT, exist_ok=True)

cols_dim = ["SK_AGENCE", "agence"] + (["DISTRICT"] if "DISTRICT" in datamart.columns else [])
datamart[cols_dim].to_csv(os.path.join(ENTREPOT, "dim_agence.csv"), index=False)
datamart[["SK_AGENCE"] + mesures].to_csv(os.path.join(ENTREPOT, "fait_agence.csv"), index=False)

if mode == "brut" and "SK_UTILISATEUR" in tables.get("objectifs", pd.DataFrame()).columns:
    obj = tables["objectifs"].dropna(subset=["SK_UTILISATEUR"]).copy()
    obj["SK_UTILISATEUR"] = obj["SK_UTILISATEUR"].astype(int)
    colonnes = [c for c in ["SK_UTILISATEUR", "LIBELLE_UTILISATEUR", "SK_AGENCE"]
                if c in tables["employes"].columns]
    dim_g = (tables["employes"][tables["employes"]["EST_GESTIONNAIRE"] == 1][colonnes]
             .drop_duplicates(subset=["SK_UTILISATEUR"]))
    fait_o = obj.groupby(["SK_AGENCE", "SK_UTILISATEUR"]).agg(
        total_comptes=("COMPTES", "sum"),
        production_credits=("CREDITS", "sum"),
        collecte_epargne=("EPARGNE", "sum")).round(1).reset_index()
    dim_g.to_csv(os.path.join(ENTREPOT, "dim_gestionnaire.csv"), index=False)
    fait_o.to_csv(os.path.join(ENTREPOT, "fait_objectif.csv"), index=False)
    print(f"axe gestionnaire : {len(dim_g)} gestionnaires, {len(fait_o)} lignes de faits")
else:
    print("axe gestionnaire non produit : la source n'a pas le détail par employé.")

datamart[["agence"] + mesures].to_csv(DATAMART, index=False)
print("entrepôt ->", ", ".join(sorted(os.listdir(ENTREPOT))))
print("datamart de segmentation ->", os.path.relpath(DATAMART, RACINE))
pd.read_csv(os.path.join(ENTREPOT, "fait_agence.csv")).head()

axe gestionnaire non produit : la source n'a pas le détail par employé.
entrepôt -> dim_agence.csv, fait_agence.csv
datamart de segmentation -> clustering/data/agences.csv


,SK_AGENCE,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
0,1,10,8,1795,799,6300000,812500
1,2,9,6,1791,607,5880000,690000
2,3,12,10,1497,557,7000000,450000
3,4,12,6,1451,618,5400000,662500
4,5,18,7,1313,708,8300000,720000


## 7. Vérification

Le notebook doit retrouver **exactement** le résultat du script
`etl/etl_agences.py`. La cellule relance le script et compare les deux
datamarts ; elle échoue si un écart apparaît.

In [9]:
import subprocess

script = os.path.join(RACINE, "etl", "etl_agences.py")
if not os.path.exists(script):
    print("Script etl/etl_agences.py absent : vérification ignorée.")
else:
    r = subprocess.run([sys.executable, script], cwd=RACINE,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-800:])
        raise RuntimeError("Le script s'est terminé en erreur.")
    identique = pd.read_csv(DATAMART).equals(
        datamart[["agence"] + mesures].reset_index(drop=True))
    print("Notebook et script produisent le même datamart :", identique)
    assert identique, "Écart entre le notebook et le script."

Notebook et script produisent le même datamart : True
